In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score


from catboost import CatBoostRegressor

In [2]:
train_sample_path = "../train_sample.csv"
test_sample_path = "../test_sample.csv"

In [3]:
train_sample = pd.read_csv(train_sample_path)
train_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor,travel_time
0,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),day,Sunday,NaN,9,1,NaN,high,NaN,1,0.878909,26.907612
1,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),morning,Thursday,NaN,7,1,medium,high,NaN,1,1.081668,27.489129


In [4]:
test_sample = pd.read_csv(test_sample_path)
test_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor
0,West Jakarta (Jakarta Barat),East Jakarta (Jakarta Timur),morning,Saturday,5.0,8,1,medium,NaN,NaN,2,1.126429
1,South Jakarta (Jakarta Selatan),East Jakarta (Jakarta Timur),evening,Saturday,NaN,9,1,low,medium,fog,2,1.121015


In [5]:
start = train_sample["start_point"].str.split().str[0]
end = train_sample["end_point"].str.split().str[0]

train_sample["start_end_point"] = start + " " + end
test_sample["start_end_point"] = test_sample["start_point"].str.split().str[0] + " " + test_sample["end_point"].str.split().str[0]

In [6]:
train_sample['start_end_point'].value_counts()

start_end_point
Central West     4153
West South       4048
Central South    4016
South East       4013
North West       4010
Central East     3987
Central North    3964
North East       3948
North South      3935
West East        3926
Name: count, dtype: int64

In [7]:
reference_cols = [
    'start_point',
    'end_point',
    'time_of_day',
    'day_of_week',
    'event_count',
    'is_holiday',
    'public_transport_availability',
    'start_end_point'
]

target_cols = [
    'traffic_condition',
    'vehicle_density',
    'population_density',
    'weather'
]

target_cols = [
    'traffic_condition',
    'vehicle_density',
    'population_density',
    'weather'
]

# Urutan level fallback
reference_levels = [
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'event_count',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day'
    ],
    [
        'start_point',
        'end_point'
    ]
]

for target in target_cols:

    for refs in reference_levels:

        # Hanya proses yang masih missing
        mask = train_sample[target].isna()

        if not mask.any():
            break

        # Cari modus berdasarkan grup
        mode_map = (
            train_sample.dropna(subset=[target])
              .groupby(refs)[target]
              .agg(lambda x: x.mode().iloc[0])
        )

        # Mapping ke baris yang masih missing
        filled = (
            train_sample.loc[mask, refs]
              .merge(
                  mode_map.rename('mode_value'),
                  left_on=refs,
                  right_index=True,
                  how='left'
              )['mode_value']
        )

        # Isi hanya yang berhasil mendapatkan modus
        train_sample.loc[mask, target] = filled.values

In [8]:
train_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
travel_time                      0
start_end_point                  0
dtype: int64

In [9]:
test_sample.isnull().sum()

start_point                        0
end_point                          0
time_of_day                        0
day_of_week                        0
traffic_condition                600
event_count                        0
is_holiday                         0
vehicle_density                  600
population_density               600
weather                          600
public_transport_availability      0
historical_delay_factor            0
start_end_point                    0
dtype: int64

In [10]:
reference_cols = [
    'start_point',
    'end_point',
    'time_of_day',
    'day_of_week',
    'event_count',
    'is_holiday',
    'public_transport_availability',
    'start_end_point'
]

target_cols = [
    'traffic_condition',
    'vehicle_density',
    'population_density',
    'weather'
]

# Urutan level fallback
reference_levels = [
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'event_count',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day'
    ],
    [
        'start_point',
        'end_point'
    ]
]

for target in target_cols:

    for refs in reference_levels:

        # Hanya proses yang masih missing
        mask = test_sample[target].isna()

        if not mask.any():
            break

        # Cari modus berdasarkan grup
        mode_map = (
            test_sample.dropna(subset=[target])
              .groupby(refs)[target]
              .agg(lambda x: x.mode().iloc[0])
        )

        # Mapping ke baris yang masih missing
        filled = (
            test_sample.loc[mask, refs]
              .merge(
                  mode_map.rename('mode_value'),
                  left_on=refs,
                  right_index=True,
                  how='left'
              )['mode_value']
        )

        # Isi hanya yang berhasil mendapatkan modus
        test_sample.loc[mask, target] = filled.values

In [11]:
test_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
start_end_point                  0
dtype: int64

In [12]:
X_test = test_sample

In [13]:
y_train = train_sample['travel_time']
X_train = train_sample[[
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "start_end_point",
    "public_transport_availability"]]
X_test = test_sample[[
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "start_end_point",
    "public_transport_availability"]]
# etc.
# your code here

In [14]:
cat_cols = [
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "start_end_point",
    "public_transport_availability"
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [15]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 13.0785634	total: 296ms	remaining: 1m 28s
100:	learn: 4.7892319	total: 4.79s	remaining: 9.44s
200:	learn: 4.7241232	total: 9.21s	remaining: 4.54s
299:	learn: 4.6890931	total: 13.5s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0942582	total: 43.9ms	remaining: 13.1s
100:	learn: 4.8014511	total: 4.18s	remaining: 8.24s
200:	learn: 4.7305656	total: 8.65s	remaining: 4.26s
299:	learn: 4.6997910	total: 12.8s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 12.9575673	total: 43.4ms	remaining: 13s
100:	learn: 4.6131774	total: 4.4s	remaining: 8.66s
200:	learn: 4.5228207	total: 9.19s	remaining: 4.53s
299:	learn: 4.4876519	total: 13.9s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0897340	total: 49.4ms	remaining: 14.8s
100:	learn: 4.7816898	total: 4.36s	remaining: 8.59s
200:	learn: 4.7078676	total: 8.69s	remaining: 4.28s
299:	learn: 4.6709497	total: 13s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0651410	total: 37.

In [16]:
model.fit(X_train, y_train)

Learning rate set to 0.195167
0:	learn: 12.9564298	total: 52.6ms	remaining: 15.7s
100:	learn: 4.7101038	total: 4.32s	remaining: 8.51s
200:	learn: 4.6444608	total: 8.82s	remaining: 4.34s
299:	learn: 4.6207507	total: 13.4s	remaining: 0us


CatBoostRegressor(cat_features=('start_point', 'end_point', 'time_of_day', 'day_of_week', 'vehicle_density', 'population_density', 'weather', 'is_holiday', 'start_end_point', 'public_transport_availability'), depth=3, iterations=300, loss_function='RMSE', random_seed=38, verbose=100)

In [17]:
y_train_hat = model.predict(X_train)
mse_lr = mean_squared_error(y_train_hat, y_train)
r2_lr = r2_score(y_train_hat, y_train)
mse_lr, r2_lr

(21.08050280989541, 0.8938590638640376)

In [18]:
y_hat_test = model.predict(X_test)
pd.DataFrame(y_hat_test).to_csv('submission.csv', index=False)